In [1]:
import pandas as pd

df = pd.read_csv('reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt

In [3]:
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

In [4]:
# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 1000  # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['clean_comment'])
y = df['category']

# Step 4: Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

In [5]:
# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)

In [6]:
from sklearn.metrics import accuracy_score, classification_report


def evaluate_trial_locally(
    model_name, model, X_train, X_test, y_train, y_test, params, trial_number
):
    """Fits the model, prints out trial performance, and returns accuracy without MLflow dependency."""
    # Train model
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)

    # Print a quick console summary for the current trial
    print(f"[Trial {trial_number}] Model: {model_name} | Accuracy: {accuracy:.4f}")
    
    # Optional: If you want to see the hyperparameter combinations being tested, uncomment below:
    # print(f" Parameters: {params}")

    return accuracy


In [7]:
# Step 6: Optuna objective function for LightGBM
def objective_lightgbm(trial):
    # Hyperparameter space to explore
    n_estimators = trial.suggest_int("n_estimators", 100, 1000)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int("max_depth", 3, 15)
    num_leaves = trial.suggest_int("num_leaves", 20, 150)
    min_child_samples = trial.suggest_int("min_child_samples", 10, 100)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    reg_alpha = trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True)  # L1 regularization
    reg_lambda = trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True)  # L2 regularization

    # Collect trial parameters into a dictionary
    params = {
        "n_estimators": n_estimators,
        "learning_rate": learning_rate,
        "max_depth": max_depth,
        "num_leaves": num_leaves,
        "min_child_samples": min_child_samples,
        "colsample_bytree": colsample_bytree,
        "subsample": subsample,
        "reg_alpha": reg_alpha,
        "reg_lambda": reg_lambda,
    }

    # Create LightGBM model
    model = LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        num_leaves=num_leaves,
        min_child_samples=min_child_samples,
        colsample_bytree=colsample_bytree,
        subsample=subsample,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=42,
    )

    # Route optimized model metadata directly to our local evaluation function
    accuracy = evaluate_trial_locally(
        "LightGBM",
        model,
        X_train,
        X_test,
        y_train,
        y_test,
        params,
        trial.number,
    )

    return accuracy


In [8]:
import optuna
from lightgbm import LGBMClassifier


# Step 7: Run Optuna for LightGBM, evaluate the best model, and plot the importance of each parameter
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=100)  # Increased to 100 trials

    # Print a summary of the hyperparameter search
    print("\n##### OPTUNA TUNING COMPLETE #####")
    print(f"Best Trial Accuracy: {study.best_value:.4f}")
    print("Best Hyperparameters Found:")
    for key, value in study.best_params.items():
        print(f"  - {key}: {value}")
    print("=" * 60 + "\n")

    # Get the best parameters
    best_params = study.best_params
    best_model = LGBMClassifier(
        n_estimators=best_params["n_estimators"],
        learning_rate=best_params["learning_rate"],
        max_depth=best_params["max_depth"],
        num_leaves=best_params["num_leaves"],
        min_child_samples=best_params["min_child_samples"],
        colsample_bytree=best_params["colsample_bytree"],
        subsample=best_params["subsample"],
        reg_alpha=best_params["reg_alpha"],
        reg_lambda=best_params["reg_lambda"],
        random_state=42,
    )

    # Route optimized model metadata directly to our local evaluation function
    evaluate_trial_locally(
        "LightGBM",
        best_model,
        X_train,
        X_test,
        y_train,
        y_test,
        best_params,
        "Best",
    )

    # Plot parameter importance
    fig_importance = optuna.visualization.plot_param_importances(study)
    fig_importance.show()

    # Plot optimization history
    fig_history = optuna.visualization.plot_optimization_history(study)
    fig_history.show()


In [ ]:
# Run the experiment for LightGBM
run_optuna_experiment()

[I 2026-10-25 22:16:23,664] A new study created in memory with name: no-name-d9104849-e8c5-439a-afaa-390ff1e480d3


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.127076 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 98781
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 956
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

[I 2026-10-25 22:17:21,517] Trial 0 finished with value: 0.6664552948636652 and parameters: {'n_estimators': 902, 'learning_rate': 0.0030363297563926464, 'max_depth': 4, 'num_leaves': 146, 'min_child_samples': 75, 'colsample_bytree': 0.8327202849084007, 'subsample': 0.6555533379909892, 'reg_alpha': 0.12006164872418817, 'reg_lambda': 0.48637110662854005}. Best is trial 0 with value: 0.6664552948636652.


[Trial 0] Model: LightGBM | Accuracy: 0.6665
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.136215 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 98978
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 966
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

[I 2026-10-25 22:22:02,624] Trial 1 finished with value: 0.7032339885859227 and parameters: {'n_estimators': 938, 'learning_rate': 0.0011095165824797425, 'max_depth': 14, 'num_leaves': 87, 'min_child_samples': 45, 'colsample_bytree': 0.8689003441701686, 'subsample': 0.5021852344697051, 'reg_alpha': 0.0005379412629803668, 'reg_lambda': 0.007113011240649965}. Best is trial 1 with value: 0.7032339885859227.


...

[Trial 51] Model: LightGBM | Accuracy: 0.8139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.163713 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 98978
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 966
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

[I 2026-10-25 23:30:06,046] Trial 52 finished with value: 0.8176918199112239 and parameters: {'n_estimators': 948, 'learning_rate': 0.06888776963813915, 'max_depth': 7, 'num_leaves': 30, 'min_child_samples': 50, 'colsample_bytree': 0.8916371579514473, 'subsample': 0.902275306117319, 'reg_alpha': 0.0022856778593337724, 'reg_lambda': 0.00034897342762806955}. Best is trial 46 with value: 0.8187486789262313.


[Trial 52] Model: LightGBM | Accuracy: 0.8177
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.241845 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 98978
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 966
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

[I 2026-10-25 23:31:57,794] Trial 53 finished with value: 0.810293806806172 and parameters: {'n_estimators': 997, 'learning_rate': 0.026629863134348664, 'max_depth': 9, 'num_leaves': 28, 'min_child_samples': 47, 'colsample_bytree': 0.8916412764747607, 'subsample': 0.9205788917477574, 'reg_alpha': 0.008891188457980986, 'reg_lambda': 0.0003733675749748432}. Best is trial 46 with value: 0.8187486789262313.


[Trial 53] Model: LightGBM | Accuracy: 0.8103
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.396151 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 98888
[LightGBM] [Info] Number of data points in the train set: 37848, number of used features: 961
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
